In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("databricks_project1").getOrCreate()

In [0]:
expenses_df = spark.read.csv(
    "/Workspace/Users/dharshinis369@gmail.com/expenses_cleaned.csv",
    header=True, inferSchema=True
)

In [0]:
expenses_df.printSchema()
print(expenses_df.count())

root
 |-- expense_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- expense_date: date (nullable = true)
 |-- description: string (nullable = true)
 |-- month: date (nullable = true)

1523


In [0]:
users_df = spark.read.csv(
    "/Workspace/Users/dharshinis369@gmail.com/users.csv",
    header=True, inferSchema=True
)
users_df.printSchema()
print(users_df.count())

root
 |-- user_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- monthly_income: integer (nullable = true)
 |-- monthly_budget: integer (nullable = true)

8


In [0]:
user_expense_df = expenses_df.join(
    users_df,
    on = "user_id",
    how = "inner"
)
user_expense_df.show()

+-------+----------+---------+-------------+-------+------------+-------------------+----------+------+----------------+------------+--------------+--------------+
|user_id|expense_id|user_name|     category| amount|expense_date|        description|     month|  name|           email|created_date|monthly_income|monthly_budget|
+-------+----------+---------+-------------+-------+------------+-------------------+----------+------+----------------+------------+--------------+--------------+
|      1|         1|     Arun|    Transport| 246.73|  2026-01-01|Metro card recharge|2026-01-01|  Arun|  arun@gmail.com|  2026-01-01|         33000|         27000|
|      2|         2|    Priya|         Food| 360.05|  2026-01-01|             Snacks|2026-01-01| Priya| priya@gmail.com|  2026-01-01|         39000|         32000|
|      2|         3|    Priya|         Food| 120.66|  2026-01-01|          Groceries|2026-01-01| Priya| priya@gmail.com|  2026-01-01|         39000|         32000|
|      2|       

In [0]:
from pyspark.sql import functions as F
user_expense_df = user_expense_df.withColumn(
    'month',
    F.date_format("expense_date", "yyyy-MM")
)
summary = user_expense_df.groupBy("user_id", "name", "month", "monthly_income", "monthly_budget").agg(
    F.sum(F.col('amount')).alias('monthly_spend')
).withColumn(
    'savings',
    F.col('monthly_income') - F.col('monthly_spend')
).withColumn("alert",
    F.when(F.col("savings") < 0, "Negative savings")
     .when(F.col("monthly_spend") > F.col("monthly_budget"), "Exceeded your budget")
     .otherwise("OK"))
summary.show()

+-------+------+-------+--------------+--------------+------------------+-------------------+----------------+
|user_id|  name|  month|monthly_income|monthly_budget|     monthly_spend|            savings|           alert|
+-------+------+-------+--------------+--------------+------------------+-------------------+----------------+
|      2| Priya|2026-01|         39000|         32000| 29597.21999999999|   9402.78000000001|              OK|
|      7|Suresh|2026-02|         27000|         22000|18875.929999999997|  8124.070000000003|              OK|
|      6| Divya|2026-03|         35000|         29000|44239.979999999996| -9239.979999999996|Negative savings|
|      3| Kavin|2026-05|         42000|         35000|26001.789999999997| 15998.210000000003|              OK|
|      8|Anitha|2026-05|         35000|         29000|28582.990000000005|  6417.009999999995|              OK|
|      3| Kavin|2026-02|         42000|         35000|27091.379999999997| 14908.620000000003|              OK|
|

In [0]:
summary.write.format("delta").mode("overwrite").saveAsTable("main.default.expense_summary")

In [0]:
%sql
SELECT * FROM main.default.expense_summary;

user_id,name,month,monthly_income,monthly_budget,monthly_spend,savings,alert
2,Priya,2026-01,39000,32000,29597.21999999999,9402.78000000001,OK
7,Suresh,2026-02,27000,22000,18875.929999999997,8124.070000000003,OK
6,Divya,2026-03,35000,29000,44239.979999999996,-9239.979999999996,Negative savings
3,Kavin,2026-05,42000,35000,26001.789999999997,15998.210000000003,OK
8,Anitha,2026-05,35000,29000,28582.990000000005,6417.009999999995,OK
3,Kavin,2026-02,42000,35000,27091.379999999997,14908.620000000003,OK
7,Suresh,2026-03,27000,22000,19796.140000000003,7203.859999999997,OK
1,Arun,2026-03,33000,27000,21743.120000000003,11256.879999999997,OK
5,Rahul,2026-04,32000,26000,23502.52999999999,8497.470000000008,OK
1,Arun,2026-06,33000,27000,25037.480000000003,7962.519999999997,OK
